# 01 — What's in the USDA livestock data?

This notebook is the answer to *"what's in this data?"* — a tour of the cleaned
observations parquet produced by `usda_sandbox.clean.clean_all`.
It loads the catalog, summarizes coverage, plots a representative sample on
shared axes, and surfaces the data-quality issues (missing months, NA spans,
month-over-month shocks) the cleaner preserved as nulls.

**Run order:** top to bottom. All cells assume `data/clean/observations.parquet`
exists. To regenerate it from the raw downloads:

```python
from usda_sandbox.ingest import sync_downloads
from usda_sandbox.clean import clean_all
sync_downloads()
clean_all("data/catalog.json", "data/raw", "data/clean/observations.parquet")
```


In [1]:
from __future__ import annotations

from pathlib import Path

import plotly.graph_objects as go
import polars as pl

from usda_sandbox.catalog import load_catalog
from usda_sandbox.store import (
    duckdb_connection,
    list_series,
    read_observations,
    read_series,
)


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from `start` (or cwd) until a directory containing pyproject.toml."""
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root (no pyproject.toml in any parent)")


PROJECT_ROOT = find_project_root()
CATALOG_PATH = PROJECT_ROOT / "data" / "catalog.json"
OBS_PATH = PROJECT_ROOT / "data" / "clean" / "observations.parquet"

pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(20)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


PROJECT_ROOT = C:\Users\tyler\usda-livestock-sandbox


## 1. Dataset overview

How big is the table, what frequencies live in it, and what date range does it
cover? These three numbers set expectations for everything downstream.

In [2]:
obs = read_observations(OBS_PATH).collect()
catalog = load_catalog(CATALOG_PATH)

print(f"Total rows:        {obs.height:,}")
print(f"Total series:      {obs['series_id'].n_unique()}")
print(f"Catalog entries:   {len(catalog)}")
print(f"Date range:        {obs['period_start'].min()}  ..  {obs['period_start'].max()}")
print(f"Total null values: {obs['value'].null_count():,}")
print()
print("Frequency mix:")
(
    obs.group_by("frequency")
       .agg(
           pl.len().alias("rows"),
           pl.col("series_id").n_unique().alias("series"),
       )
       .sort("frequency")
)


Total rows:        2,895
Total series:      12
Catalog entries:   12
Date range:        2000-01-01  ..  2026-03-01
Total null values: 113

Frequency mix:


frequency,rows,series
str,u32,u32
"""monthly""",2835,9
"""quarterly""",60,3


## 2. Series catalog

Every catalog entry, with its commodity, metric, unit, and the actual
coverage in the cleaned store.

In [3]:
series_summary = list_series(OBS_PATH)
series_summary.select(
    ["series_id", "commodity", "metric", "unit", "frequency",
     "n_obs", "n_nulls", "first_period", "last_period"]
)


series_id,commodity,metric,unit,frequency,n_obs,n_nulls,first_period,last_period
str,str,str,str,str,u32,u32,date,date
"""beef_disappearance_total_qtr""","""cattle""","""disappearance""","""million_lbs""","""quarterly""",20,0,2021-01-01,2025-10-01
"""beef_production_commercial_qtr""","""cattle""","""production""","""million_lbs""","""quarterly""",20,0,2021-01-01,2025-10-01
"""boxed_beef_cutout_choice""","""cattle""","""wholesale_price""","""USD/cwt""","""monthly""",315,0,2000-01-01,2026-03-01
"""boxed_beef_cutout_select""","""cattle""","""wholesale_price""","""USD/cwt""","""monthly""",315,0,2000-01-01,2026-03-01
"""cattle_feeder_steer_500_550""","""cattle""","""price""","""USD/cwt""","""monthly""",315,0,2000-01-01,2026-03-01
"""cattle_feeder_steer_750_800""","""cattle""","""price""","""USD/cwt""","""monthly""",315,0,2000-01-01,2026-03-01
"""cattle_steer_choice_nebraska""","""cattle""","""price""","""USD/cwt""","""monthly""",315,1,2000-01-01,2026-03-01
"""cattle_steer_choice_tx_ok_nm""","""cattle""","""price""","""USD/cwt""","""monthly""",315,4,2000-01-01,2026-03-01
"""hog_barrow_gilt_natbase_51_52""","""hogs""","""price""","""USD/cwt""","""monthly""",315,12,2000-01-01,2026-03-01


### 2a. By commodity

DuckDB SQL through the `obs` view — same data, different lens.

In [4]:
con = duckdb_connection(OBS_PATH)
try:
    by_commodity = con.execute("""
        SELECT commodity,
               COUNT(DISTINCT series_id)                          AS n_series,
               COUNT(*)                                            AS n_rows,
               SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END)      AS n_nulls,
               MIN(period_start)                                   AS earliest,
               MAX(period_start)                                   AS latest
        FROM obs
        GROUP BY commodity
        ORDER BY commodity
    """).pl()
finally:
    con.close()
by_commodity


commodity,n_series,n_rows,n_nulls,earliest,latest
str,i64,i64,"decimal[38,0]",date,date
"""cattle""",8,1930,5,2000-01-01,2026-03-01
"""hogs""",3,650,12,2000-01-01,2026-03-01
"""sheep_lamb""",1,315,96,2000-01-01,2026-03-01


## 3. A representative sample on shared axes

The seven `USD/cwt` price series share a unit, so they go on a single
y-axis — relative scales are visible directly. Cattle prices anchor the top of
the chart, hogs and lambs the bottom; gaps show where the source published
`NA` (the cleaner preserved them as nulls and `connectgaps=False` makes them
visible).

Quarterly `million_lbs` production series are scaled differently and live in
their own chart.

In [5]:
price_ids = (
    series_summary
    .filter(pl.col("unit") == "USD/cwt")
    .sort("series_id")["series_id"]
    .to_list()
)

fig = go.Figure()
for sid in price_ids:
    s = read_series(sid, OBS_PATH)
    fig.add_trace(
        go.Scatter(
            x=s["period_start"].to_list(),
            y=s["value"].to_list(),
            name=s["series_name"][0],
            mode="lines",
            connectgaps=False,
        )
    )

fig.update_layout(
    title="USDA livestock prices (monthly) — USD/cwt, shared axes",
    xaxis_title="Period",
    yaxis_title="USD per hundredweight",
    height=600,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.45, xanchor="left", x=0),
)
fig.show()


In [6]:
quarterly_ids = (
    series_summary
    .filter(pl.col("frequency") == "quarterly")
    .sort("series_id")["series_id"]
    .to_list()
)

fig = go.Figure()
for sid in quarterly_ids:
    s = read_series(sid, OBS_PATH)
    fig.add_trace(
        go.Scatter(
            x=s["period_start"].to_list(),
            y=s["value"].to_list(),
            name=s["series_name"][0],
            mode="lines+markers",
        )
    )

fig.update_layout(
    title="WASDE quarterly supply / disappearance — million lbs",
    xaxis_title="Quarter start",
    yaxis_title="Million pounds",
    height=420,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="left", x=0),
)
fig.show()


## 4. Data quality

Three checks the cleaner is responsible for, all spot-checked here:

1. **Missing rows** — every monthly series should have a row for every month
   between its first and last observation. (Nulls are a separate concern,
   covered next.)
2. **Null spans** — values surfaced as null mean the source itself reported
   no value (`NA`, `--`, blank). Where these cluster matters more than the
   raw count.
3. **Month-over-month shocks** — large jumps that aren't units bugs are
   real economic events. Flagging them lets us spot-check.
4. **Unit consistency** — no series should change units mid-stream.

In [7]:
# 4.1 — Missing rows in monthly series.
def gaps_for(series_id: str) -> list:
    s = read_series(series_id, OBS_PATH)
    if s.is_empty():
        return []
    expected = pl.date_range(
        start=s["period_start"].min(),
        end=s["period_start"].max(),
        interval="1mo",
        eager=True,
    ).to_list()
    actual = set(s["period_start"].to_list())
    return [d for d in expected if d not in actual]


monthly_ids = series_summary.filter(pl.col("frequency") == "monthly")["series_id"].to_list()
missing_rows: dict[str, int] = {sid: len(gaps_for(sid)) for sid in monthly_ids}
print("Missing months by series (rows where the source skipped a period entirely):")
for sid, n in missing_rows.items():
    print(f"  {sid:<40} {n} missing")


Missing months by series (rows where the source skipped a period entirely):
  boxed_beef_cutout_choice                 0 missing
  boxed_beef_cutout_select                 0 missing
  cattle_feeder_steer_500_550              0 missing
  cattle_feeder_steer_750_800              0 missing
  cattle_steer_choice_nebraska             0 missing
  cattle_steer_choice_tx_ok_nm             0 missing
  hog_barrow_gilt_natbase_51_52            0 missing
  lamb_slaughter_choice_san_angelo         0 missing
  pork_cutout_composite                    0 missing


In [8]:
# 4.2 — Where are the null spans?
null_summary = (
    obs.filter(pl.col("value").is_null())
       .group_by("series_id")
       .agg(
           pl.len().alias("n_nulls"),
           pl.col("period_start").min().alias("first_null"),
           pl.col("period_start").max().alias("last_null"),
       )
       .sort("n_nulls", descending=True)
)
null_summary


series_id,n_nulls,first_null,last_null
str,u32,date,date
"""lamb_slaughter_choice_san_ange…",96,2018-04-01,2026-03-01
"""hog_barrow_gilt_natbase_51_52""",12,2025-04-01,2026-03-01
"""cattle_steer_choice_tx_ok_nm""",4,2019-11-01,2025-12-01
"""cattle_steer_choice_nebraska""",1,2022-01-01,2022-01-01


In [9]:
# 4.3 — Month-over-month shocks (>25% absolute change).
shocks = (
    obs.filter((pl.col("frequency") == "monthly") & pl.col("value").is_not_null())
       .sort(["series_id", "period_start"])
       .with_columns(prev=pl.col("value").shift(1).over("series_id"))
       .with_columns(
           pct_change=((pl.col("value") - pl.col("prev")) / pl.col("prev") * 100).round(1)
       )
       .filter(pl.col("pct_change").abs() > 25)
       .select(["series_id", "period_start", "prev", "value", "pct_change"])
       .sort(["series_id", "period_start"])
)
print(f"Monthly observations with >25% MoM change: {shocks.height}")
shocks.head(30)


Monthly observations with >25% MoM change: 17


series_id,period_start,prev,value,pct_change
str,date,f64,f64,f64
"""boxed_beef_cutout_choice""",2020-05-01,241.94,405.8,67.7
"""boxed_beef_cutout_choice""",2020-06-01,405.8,242.31,-40.3
"""boxed_beef_cutout_select""",2020-05-01,229.65,386.24,68.2
"""boxed_beef_cutout_select""",2020-06-01,386.24,228.0,-41.0
"""cattle_feeder_steer_500_550""",2013-02-01,134.75,178.88,32.7
"""hog_barrow_gilt_natbase_51_52""",2008-05-01,45.0586,57.7496,28.2
"""hog_barrow_gilt_natbase_51_52""",2014-03-01,63.8768,83.8198,31.2
"""hog_barrow_gilt_natbase_51_52""",2015-05-01,45.1326,57.2316,26.8
"""hog_barrow_gilt_natbase_51_52""",2018-08-01,55.38,37.88,-31.6


In [10]:
# 4.4 — Unit consistency: no series should mix units.
unit_changes = (
    obs.group_by("series_id")
       .agg(pl.col("unit").n_unique().alias("n_units"))
       .filter(pl.col("n_units") > 1)
)
if unit_changes.is_empty():
    print("OK — every series uses a single unit throughout.")
else:
    print("WARNING — series with mid-stream unit changes:")
    print(unit_changes)


OK — every series uses a single unit throughout.


## Takeaway

- The store currently holds **12 series and ~2.9k observations** spanning
  **2000-01 through 2026-03** (monthly price/wholesale series), plus a
  smaller window of **quarterly WASDE supply/disappearance** since 2021.
- Monthly coverage has **no missing rows** — every series has an entry for
  every month between its first and last observation. Where data is genuinely
  unavailable, the cleaner emits a row with `value = NULL` rather than
  silently dropping it.
- The bulk of the nulls are concentrated in two known reporting issues:
  the **lamb slaughter San Angelo** series effectively went dark after 2018,
  and **hog 51-52% lean** has gaps in recent months (the live release isn't
  populated yet for everything). These will need a substitution strategy
  before forecasting them.
- Month-over-month shocks line up with real events (cattle-cycle peaks,
  COVID-era pricing) rather than parsing artifacts. No false positives from
  unit issues.
- **No series changes units mid-stream**, so downstream forecasting code can
  treat each `series_id` as a single consistent variable.

Next: `02_visualize.ipynb` for time-domain treatment (decomposition, YoY,
correlations); `03_forecast.ipynb` for the model bake-off.